# Topic Graph Exploration

End-to-end k-means + k-NN + 3D force-directed pipeline over the paper corpus.
Mirrors the artifact served by `/api/topic-map` and previews it as an interactive 3D plotly graph.

The main 3D view uses a regular `go.Figure` so it renders in any notebook viewer (GitHub, VS Code preview, nbviewer) without needing a live kernel. A separate optional cell below provides live label editing via widgets — that one requires a running Jupyter kernel.


## Setup

Load the pre-built embeddings (`abstracts.npy`) and the paper index. Mirrors `notebooks/exploration.ipynb`.

In [1]:
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import config
from src.clustering import assign_topic_labels, kmeans
from src.graph import knn_graph
from src.data import load_corpus

EMBEDDINGS_DIR = config.EMBEDDINGS_DIR
PROCESSED_DIR  = config.PROCESSED_DIR

abstracts = np.load(os.path.join(EMBEDDINGS_DIR, "abstracts.npy"))
with open(os.path.join(EMBEDDINGS_DIR, "index.json")) as f:
    index = json.load(f)
abstract_ids = index["abstracts"]

papers_path = os.path.join(PROCESSED_DIR, "papers.json")
papers = load_corpus(papers_path) if os.path.exists(papers_path) else []
paper_by_id = {p.paper_id: p for p in papers}
ordered_papers = [paper_by_id.get(pid) for pid in abstract_ids]

print(f"Embeddings : {abstracts.shape}")
print(f"Encoder    : {index.get('encoder_model', 'unknown')}")
print(f"Papers     : {sum(p is not None for p in ordered_papers)} / {len(abstract_ids)} aligned")

Embeddings : (273, 768)
Encoder    : all-mpnet-base-v2
Papers     : 273 / 273 aligned


## Run k-means

Lloyd's algorithm + k-means++ init + spherical renormalization. Pulls defaults from `config` (k came from the data-driven NMI-vs-arxiv-category sweep — see the last cell).

In [2]:
centroids, assignments, inertia = kmeans(
    abstracts,
    k=config.N_CLUSTERS,
    max_iter=config.KMEANS_MAX_ITER,
    seed=config.KMEANS_SEED,
    n_init=10,
)

sizes = [int((assignments == j).sum()) for j in range(config.N_CLUSTERS)]
print(f"k           = {config.N_CLUSTERS}")
print(f"inertia     = {inertia:.3f}")
print(f"sizes       = {sizes}")

k           = 13
inertia     = 194.877
sizes       = [13, 24, 15, 9, 28, 23, 39, 23, 24, 19, 20, 23, 13]


### Silhouette score (pure numpy, O(N²))

In [3]:
def silhouette_score(X: np.ndarray, assignments: np.ndarray) -> float:
    N = X.shape[0]
    x_sq = np.sum(X * X, axis=1, keepdims=True)
    dists = np.sqrt(np.maximum(x_sq + x_sq.T - 2 * X @ X.T, 0.0))
    scores = np.zeros(N)
    clusters = np.unique(assignments)
    for i in range(N):
        own = assignments[i]
        same = assignments == own
        same[i] = False
        a = dists[i, same].mean() if same.any() else 0.0
        b_candidates = []
        for c in clusters:
            if c == own:
                continue
            mask = assignments == c
            if mask.any():
                b_candidates.append(dists[i, mask].mean())
        if not b_candidates:
            continue
        b = min(b_candidates)
        scores[i] = (b - a) / max(a, b) if max(a, b) > 0 else 0.0
    return float(scores.mean())

print(f"silhouette = {silhouette_score(abstracts, assignments):.4f}")

silhouette = 0.0366


## Build the k-NN graph

Undirected cosine-similarity k-NN, symmetrized. Edge weight = cosine similarity.

In [4]:
edges = knn_graph(abstracts, k_neighbors=config.KNN_NEIGHBORS)
print(f"{len(edges)} undirected edges")

degree = np.zeros(abstracts.shape[0], dtype=np.int64)
for i, j, _ in edges:
    degree[i] += 1
    degree[j] += 1

print(f"degree: min={degree.min()}, mean={degree.mean():.2f}, max={degree.max()}")

1596 undirected edges
degree: min=8, mean=11.69, max=28


## Generate cluster labels

Toggle `USE_LLM` between the flan-T5 labeler and the closest-paper-title fallback. LLM mode uses few-shot prompting to coax coherent labels out of the small model.

In [5]:
USE_LLM = True  # flan-T5 few-shot labeling; flip to False for deterministic title fallback

llm_mod = None
if USE_LLM:
    # Silence transformers tqdm so weight-loading progress bars don't leak
    # widget state into the notebook (breaks static viewers without a kernel).
    from transformers.utils import logging as _tf_logging
    _tf_logging.disable_progress_bar()
    import src.llm as llm_mod

labels = assign_topic_labels(
    X=abstracts,
    centroids=centroids,
    papers=ordered_papers,
    assignments=assignments,
    llm=llm_mod,
)

for j, label in enumerate(labels):
    print(f"cluster {j} ({sizes[j]:3d} papers): {label}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


cluster 0 ( 13 papers): weather forecasting
cluster 1 ( 24 papers): neural network optimization
cluster 2 ( 15 papers): medical ai
cluster 3 (  9 papers): LLM cognition
cluster 4 ( 28 papers): multimodal reasoning
cluster 5 ( 23 papers): subjectivity and bias detection
cluster 6 ( 39 papers): reasoning models
cluster 7 ( 23 papers): transformer model compression
cluster 8 ( 24 papers): machine learning
cluster 9 ( 19 papers): reinforcement learning
cluster 10 ( 20 papers): reinforcement learning
cluster 11 ( 23 papers): federated learning
cluster 12 ( 13 papers): traffic simulation and planning


## 3D force-directed layout

Fruchterman-Reingold in 3D (pure numpy). Repulsion between all nodes, attraction along k-NN edges, linear cooling. Produces coordinates in a roughly unit-radius cube.

In [6]:
def force_directed_3d(
    N: int,
    edges: list,
    iterations: int = 200,
    seed: int = 42,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    k = (1.0 / N) ** (1.0 / 3.0)
    pos = rng.uniform(-0.5, 0.5, size=(N, 3))
    edge_arr = np.array([[a, b] for a, b, _ in edges], dtype=np.int64) if edges else np.empty((0, 2), dtype=np.int64)
    edge_w  = np.array([w for _, _, w in edges], dtype=np.float64) if edges else np.empty(0)
    temperature = 0.1

    for it in range(iterations):
        delta = pos[:, None, :] - pos[None, :, :]
        dist = np.linalg.norm(delta, axis=2) + 1e-9
        rep_mag = (k * k) / dist
        np.fill_diagonal(rep_mag, 0.0)
        rep = (delta / dist[..., None]) * rep_mag[..., None]
        forces = rep.sum(axis=1)

        if len(edge_arr) > 0:
            a_idx = edge_arr[:, 0]
            b_idx = edge_arr[:, 1]
            d_ab = pos[a_idx] - pos[b_idx]
            l_ab = np.linalg.norm(d_ab, axis=1, keepdims=True) + 1e-9
            w = np.maximum(edge_w, 0.0)[:, None]
            att_mag = (l_ab * l_ab) / k
            att = (d_ab / l_ab) * att_mag * w
            np.add.at(forces, a_idx, -att)
            np.add.at(forces, b_idx,  att)

        fmag = np.linalg.norm(forces, axis=1, keepdims=True) + 1e-9
        step = forces / fmag * np.minimum(fmag, temperature)
        pos = pos + step
        temperature *= 0.97

    return pos

positions = force_directed_3d(abstracts.shape[0], edges, iterations=200, seed=42)
print(f"positions: {positions.shape}, extent: {positions.min():.3f} to {positions.max():.3f}")

positions: (273, 3), extent: -1.465 to 1.402


## Interactive 3D view (static-renderable)

Plotly WebGL scatter: **rotate** (left-drag), **zoom** (scroll), **pan** (right-drag). Hover any node to see its paper title and cluster. Click a cluster name in the legend to hide/show.

This cell uses a regular `go.Figure` so the embedded HTML renders in any notebook viewer — no live kernel required. To edit cluster labels, use the optional widget cell below (requires a running kernel).

In [7]:
import plotly.graph_objects as go

def cluster_color(i: int, total: int) -> str:
    hue = round((i * 360) / max(total, 1))
    return f"hsl({hue}, 70%, 55%)"


def build_topic_figure(labels_list):
    fig = go.Figure()

    # Edge trace (single polyline with None breaks between segments)
    ex, ey, ez = [], [], []
    for a, b, _w in edges:
        ex += [positions[a, 0], positions[b, 0], None]
        ey += [positions[a, 1], positions[b, 1], None]
        ez += [positions[a, 2], positions[b, 2], None]
    fig.add_trace(go.Scatter3d(
        x=ex, y=ey, z=ez,
        mode="lines",
        line=dict(color="rgba(150,150,150,0.18)", width=1),
        hoverinfo="skip",
        showlegend=False,
        name="edges",
    ))

    # One node trace per cluster so the legend controls per-cluster visibility
    for j in range(config.N_CLUSTERS):
        mask = assignments == j
        if not mask.any():
            continue
        rows = np.where(mask)[0]
        titles = [ordered_papers[int(r)].title if ordered_papers[int(r)] else abstract_ids[int(r)] for r in rows]
        fig.add_trace(go.Scatter3d(
            x=positions[mask, 0], y=positions[mask, 1], z=positions[mask, 2],
            mode="markers",
            marker=dict(size=4.5, color=cluster_color(j, config.N_CLUSTERS), opacity=0.9),
            text=titles,
            customdata=[labels_list[j]] * int(mask.sum()),
            hovertemplate="<b>%{customdata}</b><br>%{text}<extra></extra>",
            name=f"{j}: {labels_list[j]}",
        ))

    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="#0b0f14",
        ),
        paper_bgcolor="#0b0f14",
        font=dict(color="#e5e7eb"),
        legend=dict(bgcolor="rgba(0,0,0,0)", itemsizing="constant"),
        margin=dict(l=0, r=0, t=10, b=0),
        height=640,
        title=f"{abstracts.shape[0]} papers · k={config.N_CLUSTERS} · {len(edges)} k-NN edges",
    )
    return fig

fig = build_topic_figure(labels)
fig.show()

### Optional: live label editor

**Requires a running Jupyter kernel.** This cell creates a text input per cluster. After you type new names, click **Apply & re-render** to rebuild the 3D plot with the updated labels and refresh hover tooltips. The `labels` list is mutated in place, so the export cell below picks up your edits.

In static viewers (GitHub, nbviewer, VS Code preview without a kernel) this cell will be blank or show a widget placeholder — that's expected. The static 3D plot above still works.

In [8]:
from IPython.display import display, clear_output
from ipywidgets import Text, VBox, Label, Button, Layout, Output

_label_widgets = [
    Text(value=labels[j], description=f"{j}:", layout=Layout(width="520px"))
    for j in range(config.N_CLUSTERS)
]
_apply_btn = Button(description="Apply & re-render", button_style="primary")
_plot_out = Output()

def _on_apply(_):
    for j, w in enumerate(_label_widgets):
        labels[j] = w.value
    with _plot_out:
        clear_output(wait=True)
        build_topic_figure(labels).show()

_apply_btn.on_click(_on_apply)

display(VBox([
    Label("Rename clusters — click Apply to rebuild the 3D plot with the new labels:"),
    *_label_widgets,
    _apply_btn,
    _plot_out,
]))

## Top papers per cluster

Manual sanity check: the 5 papers closest to each centroid. Use this to judge whether the LLM labels make sense.

In [9]:
for j in range(config.N_CLUSTERS):
    member_idx = np.where(assignments == j)[0]
    if len(member_idx) == 0:
        print(f"\n--- Cluster {j}: {labels[j]} (empty) ---")
        continue
    diffs = abstracts[member_idx] - centroids[j]
    sq = np.einsum("ij,ij->i", diffs, diffs)
    order = np.argsort(sq)[:5]
    print(f"\n--- Cluster {j}: {labels[j]} ({len(member_idx)} papers) ---")
    for rank, idx in enumerate(order, 1):
        paper = ordered_papers[int(member_idx[idx])]
        if paper is None:
            continue
        print(f"  {rank}. [{sq[idx]:.4f}]  {paper.title}")


--- Cluster 0: weather forecasting (13 papers) ---
  1. [0.4357]  StretchCast: Global-Regional AI Weather Forecasting on Stretched Cubed-Sphere Mesh
  2. [0.5472]  Skillful Kilometer-Scale Regional Weather Forecasting via Global and Regional Coupling
  3. [0.6192]  Physics-Guided Transformer (PGT): Physics-Aware Attention Mechanism for PINNs
  4. [0.6297]  RG-TTA: Regime-Guided Meta-Control for Test-Time Adaptation in Streaming Time Series
  5. [0.6750]  MR-ImagenTime: Multi-Resolution Time Series Generation through Dual Image Representations

--- Cluster 1: neural network optimization (24 papers) ---
  1. [0.5071]  Beta-Scheduling: Momentum from Critical Damping as a Diagnostic and Correction Tool for Neural Network Training
  2. [0.5305]  Universal Approximation Constraints of Narrow ResNets: The Tunnel Effect
  3. [0.5933]  The Unreasonable Effectiveness of Scaling Laws in AI
  4. [0.6035]  The Geometric Cost of Normalization: Affine Bounds on the Bayesian Complexity of Neural Netw

## Export edited labels back to `topic_graph.json`

If you renamed clusters above (either by editing the `labels` list directly or via the widget editor), run this cell to rewrite `data/processed/topic_graph.json`. Nodes, edges, and meta are preserved.

In [10]:
import json

target = os.path.join(PROCESSED_DIR, "topic_graph.json")
if os.path.exists(target):
    with open(target) as f:
        artifact = json.load(f)
    for c in artifact["clusters"]:
        c["label"] = labels[c["id"]]
    with open(target, "w") as f:
        json.dump(artifact, f, indent=2)
    print(f"Updated labels in {target}")
else:
    print(f"No artifact at {target}. Run scripts/compute_topic_graph.py first.")

Updated labels in /mnt/c/Users/xanca/Repositories/CSCI473_Project/notebooks/../data/processed/topic_graph.json


## k sweep (elbow, silhouette, arXiv-category agreement)

Validates the `N_CLUSTERS` choice. Silhouette is weak on 768-d sentence-transformer embeddings so we also compare clusters against each paper's arXiv `primary_category` via ARI and NMI. NMI rises with k; the chosen k is the largest non-degenerate value before singleton clusters appear.

In [11]:
from math import log

# Load primary_category per paper from the raw enriched JSON files
primary = {}
raw_dir = os.path.join("..", "data", "raw", "enriched")
for fn in os.listdir(raw_dir):
    if not fn.endswith(".json"):
        continue
    with open(os.path.join(raw_dir, fn)) as f:
        p = json.load(f)
    primary[p["paper_id"]] = p.get("primary_category", "")
pri_labels = [primary.get(pid, "") for pid in abstract_ids]
pri_to_int = {c: i for i, c in enumerate(sorted(set(pri_labels)))}
pri_int = np.array([pri_to_int[c] for c in pri_labels])

def _contingency(a, b):
    ca, cb = np.unique(a), np.unique(b)
    M = np.zeros((len(ca), len(cb)), dtype=np.int64)
    ai = {v: i for i, v in enumerate(ca)}
    bi = {v: i for i, v in enumerate(cb)}
    for x, y in zip(a, b):
        M[ai[x], bi[y]] += 1
    return M

def _c2(x): return x * (x - 1) // 2

def ari(a, b):
    M = _contingency(a, b)
    n = M.sum()
    s_ij = sum(_c2(int(v)) for v in M.flatten())
    s_a  = sum(_c2(int(v)) for v in M.sum(axis=1))
    s_b  = sum(_c2(int(v)) for v in M.sum(axis=0))
    exp  = s_a * s_b / _c2(n)
    mx   = 0.5 * (s_a + s_b)
    return 0.0 if mx == exp else (s_ij - exp) / (mx - exp)

def nmi(a, b):
    M = _contingency(a, b).astype(np.float64)
    n = M.sum()
    P = M / n
    pa = P.sum(axis=1); pb = P.sum(axis=0)
    mi = 0.0
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            if P[i, j] > 0 and pa[i] > 0 and pb[j] > 0:
                mi += P[i, j] * log(P[i, j] / (pa[i] * pb[j]))
    h_a = -sum(p * log(p) for p in pa if p > 0)
    h_b = -sum(p * log(p) for p in pb if p > 0)
    denom = 0.5 * (h_a + h_b)
    return 0.0 if denom == 0 else mi / denom

k_values  = list(range(4, 31))
inertias  = []
sils      = []
aris      = []
nmis      = []
min_sizes = []
for k in k_values:
    _, asg, ine = kmeans(abstracts, k=k, seed=config.KMEANS_SEED, n_init=10)
    inertias.append(ine)
    sils.append(silhouette_score(abstracts, asg))
    aris.append(ari(asg, pri_int))
    nmis.append(nmi(asg, pri_int))
    min_sizes.append(min(int((asg == j).sum()) for j in range(k)))

from plotly.subplots import make_subplots
fig_sweep = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Inertia (elbow)", "Silhouette", "Agreement with arXiv primary_category"),
)
fig_sweep.add_trace(go.Scatter(x=k_values, y=inertias, mode="lines+markers", name="inertia"),     row=1, col=1)
fig_sweep.add_trace(go.Scatter(x=k_values, y=sils,     mode="lines+markers", name="silhouette"),  row=1, col=2)
fig_sweep.add_trace(go.Scatter(x=k_values, y=aris,     mode="lines+markers", name="ARI"),         row=1, col=3)
fig_sweep.add_trace(go.Scatter(x=k_values, y=nmis,     mode="lines+markers", name="NMI"),         row=1, col=3)
fig_sweep.update_layout(
    height=360,
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#e5e7eb"),
    showlegend=True,
)
fig_sweep.update_xaxes(title="k", gridcolor="#2a3038")
fig_sweep.update_yaxes(gridcolor="#2a3038")
fig_sweep.show()

print(f"{'k':>3}  {'inertia':>8}  {'sil':>7}  {'ARI':>7}  {'NMI':>6}  {'min':>4}  note")
for k, i, s, a, n, m in zip(k_values, inertias, sils, aris, nmis, min_sizes):
    marker = " ← current" if k == config.N_CLUSTERS else ""
    note = "singleton!" if m <= 2 else ""
    print(f"{k:>3}  {i:>8.3f}  {s:>7.4f}  {a:>+7.4f}  {n:>6.4f}  {m:>4d}  {note}{marker}")

  k   inertia      sil      ARI     NMI   min  note
  4   228.543   0.0449  +0.2007  0.3045    43  
  5   221.421   0.0460  +0.1579  0.3090    36  
  6   216.149   0.0407  +0.1219  0.2795    26  
  7   212.183   0.0421  +0.1213  0.3258    23  
  8   208.885   0.0399  +0.0975  0.3066    22  
  9   205.840   0.0371  +0.0888  0.3222    15  
 10   202.437   0.0379  +0.0863  0.3345    12  
 11   199.880   0.0365  +0.0764  0.3322    12  
 12   198.230   0.0398  +0.1046  0.3466     2  singleton!
 13   194.877   0.0366  +0.0852  0.3435     9   ← current
 14   192.830   0.0456  +0.0839  0.3461     1  singleton!
 15   190.196   0.0447  +0.0727  0.3481     1  singleton!
 16   188.172   0.0377  +0.0639  0.3644     5  
 17   185.924   0.0365  +0.0713  0.3430     4  
 18   183.203   0.0406  +0.0569  0.3614     5  
 19   181.019   0.0411  +0.0545  0.3681     5  
 20   180.307   0.0339  +0.0659  0.3587     4  
 21   179.174   0.0363  +0.0517  0.3669     3  
 22   177.021   0.0344  +0.0572  0.3589     